# Function 8 · Week 10 decision review

Eight-input separable score surface

[Detailed observations](observation_review.ipynb) · [Detailed fits](model_predictions.ipynb) · [Previous pooled study](full-cohort/model_predictions.ipynb) · [2024 comparison](../../../side-quests/historical-cohort-2024/README.md) · [Course data-use review](../../../docs/capstone-data-policy.md)

**Decision status.** Week 10 is prepared and unsubmitted. Round 9 is the latest official return. Start with the [prepared comparison](#prepared-week-10-decision) when choosing between the two portal batches.

## Expectations

Round 9 evaluated a Matérn compromise that changed all eight coordinates. The earlier archived GP replay is shown for comparability. Its uncertainty reflects the saved weekly model.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

from IPython.display import display
import numpy as np
import pandas as pd

FUNCTION = 8
WEEK_DIR = Path.cwd().resolve()
WEEK = int(WEEK_DIR.name.removeprefix("Week_"))
STAGE = WEEK_DIR.parents[2]
SHARED = STAGE / "notebooks/Shared"
SPECIAL = STAGE / "notebooks/Function_1/FunctionSpecificShared"
sys.path[:0] = [str(SHARED), str(SPECIAL)]
from weekly_peak_review import peak_review, portal_point
from query_budget import remaining_queries

analysis = WEEK_DIR / "official" if FUNCTION <= 4 else WEEK_DIR
official_path = analysis / "observations.csv"
model_snapshot = json.loads((analysis / "model-review.json").read_bytes())
observation_snapshot = json.loads((analysis / "observation-review.json").read_bytes())
feedback = json.loads((STAGE / f"submissions/round-{WEEK-1:02d}-results.json").read_bytes())
record = feedback["results"][FUNCTION - 1]
forecast = observation_snapshot["historical_forecast"]
if (model_snapshot["week"], model_snapshot["function"], model_snapshot["snapshot_sha256"]) != (
        WEEK, FUNCTION, sha256(official_path.read_bytes()).hexdigest()):
    raise ValueError("The official model review and observations describe different snapshots")
if (feedback["round"], record["function"], record["portal_input"], record["output"]) != (
        WEEK - 1, FUNCTION, forecast["portal_input"], forecast["actual_output"]):
    raise ValueError("The saved forecast must match the returned observation")
official = pd.read_csv(official_path, float_precision="round_trip")
cols = [c for c in official if c.startswith("x")]
x = official[cols].to_numpy(float)
raw_y = official.y.to_numpy(float)
later = official.source_round.to_numpy() > 0
best_row = official.loc[official.y.idxmax()]
incumbent = best_row[cols].to_numpy(float)
queries_left = remaining_queries(WEEK_DIR)
units = forecast["prediction_units"]
target = raw_y
if FUNCTION == 1:
    from function_1_model_helpers import signed_log
    target = signed_log(raw_y, scale=forecast["height_scale"])

display(pd.Series({
    "Archived forecast source": forecast["source"],
    "Submitted input": forecast["portal_input"],
    "Forecast units": units,
    "Saved GP mean": forecast["gp_mean"],
    "GP latent SD": forecast["gp_latent_sd"],
    "Saved neural mean": forecast["neural_mean"],
    "Neural seed SD": forecast["neural_seed_sd"],
}, name="Forecast frozen before the return"))

Archived forecast source    notebooks/Function_8/Week_09/figures/neural-ne...
Submitted input             0.000000-0.365002-0.020186-0.150000-0.796758-0...
Forecast units                                                       output y
Saved GP mean                                                        9.996644
GP latent SD                                                         0.191627
Saved neural mean                                                    9.520305
Neural seed SD                                                       0.017406
Name: Forecast frozen before the return, dtype: object

## Results

The round 9 point scored 9.87427. Aby’s faster gains came from exploiting the useful starter region and then trusting a GP whose forecasts had held up. The archive provides near-10 measurements and an independent return of 10 at the centre recovered below.

In [2]:
latest_fitted = forecast["actual_in_prediction_units"]
display(pd.Series({"Returned score": record["output"], "Best before return": record["best_before"],
                   "Best now": record["best_after"], "Return rank": observation_snapshot["latest_rank"],
                   "Actual in forecast units": latest_fitted,
                   "Actual minus GP mean": forecast["actual_minus_gp_mean"],
                   "Actual minus neural mean": forecast["actual_minus_neural_mean"]}, name="Official result"))

# These references describe observed outcomes. The decision fits use official rows.
def optional_reference(folder, file, audit_name, expected):
    path = folder / file
    manifest = folder / audit_name
    if not (path.is_file() and manifest.is_file()):
        return None
    manifest_data = json.loads(manifest.read_bytes())
    if sha256(path.read_bytes()).hexdigest() != expected(manifest_data):
        raise ValueError(f"Reference hash mismatch: {path}")
    return pd.read_csv(path, float_precision="round_trip")

current_reference = optional_reference(
    STAGE / "data/cohort/2026-11-25", f"function_{FUNCTION}/observations.csv", "dataset-audit.json",
    lambda a: a["summary"][FUNCTION - 1]["sha256"],
)
archive = optional_reference(
    STAGE / "data/historical-cohort/2024", f"function_{FUNCTION}.csv", "intake-audit.json",
    lambda a: a["functions"][str(FUNCTION)]["sha256"],
)
reference_rows = []
if current_reference is not None:
    peer = current_reference.loc[current_reference.observation_id.str.startswith("peer-")]
    reference_rows.append(("Published current cohort", peer))
if archive is not None:
    reference_rows.append(("2024 archive, including extra starters", archive))
benchmarks = []
for label, frame in reference_rows:
    row = frame.loc[frame.y.idxmax()]
    benchmarks.append({"Scope": label, "Highest measured": row.y,
                       "Difference from your best": row.y - best_row.y,
                       "Distance to your best": np.linalg.norm(row[cols].to_numpy(float) - incumbent)})
if benchmarks:
    display(pd.DataFrame(benchmarks).round(8))
else:
    print("Local comparison snapshots are unavailable for this run.")

Returned score              9.874267
Best before return          9.936773
Best now                    9.936773
Return rank                 3.000000
Actual in forecast units    9.874267
Actual minus GP mean       -0.122376
Actual minus neural mean    0.353962
Name: Official result, dtype: float64

,Scope,Highest measured,Difference from your best,Distance to your best
0,Published current cohort,9.946588,0.009814,0.243141
1,"2024 archive, including extra starters",9.999944,0.063170,0.274464


In [3]:
from weekly_peak_review import separable_quadratic
quadratic = separable_quadratic(x, raw_y, later)
print(f"Fitted only on {quadratic['starter_rows']} starter rows.")
print(f"Maximum absolute error on all {quadratic['later_rows']} subsequent returns: {quadratic['later_max_abs_error']:.3g}.")
print("Positive loss weights around the stationary point:", np.round(quadratic["curvature"], 8))
print(f"Analytic maximum on the fitted surface: {quadratic['predicted_maximum']:.12f}.")
print(f"Six-decimal maximiser: {quadratic['portal_input']}.")
print(f"Difference from your measured incumbent: {quadratic['predicted_maximum'] - best_row.y:.8f}.")
assert not np.any(np.all(x.round(6) == np.array(quadratic["point"]), axis=1))

Fitted only on 40 starter rows.
Maximum absolute error on all 9 subsequent returns: 1.95e-14.
Positive loss weights around the stationary point: [2.  1.  3.  1.  0.5 1.  2.  0.1]
Analytic maximum on the fitted surface: 10.000000000000.
Six-decimal maximiser: 0.100000-0.150000-0.130000-0.150000-0.800000-0.500000-0.200000-0.600000.
Difference from your measured incumbent: 0.06322658.


## Predictions

**Local peak and global evidence**

Your incumbent is about 0.06323 below the fitted maximum and outside the 0.01 tolerance. The starter-only separable quadratic has full rank, is strictly concave, and predicts all nine later returns to numerical precision. The stationary point is its unique global maximum and lies on the portal lattice. A hidden term that vanishes on every tested point is still possible for an arbitrary black box.

### Prepared Week 10 decision

**Prepared and unsubmitted as of 25 November 2026.** Round 9 is the latest confirmed feedback. Four portal queries remain per function, including Week 10. Use the cohort-informed route only if the course permits that use, including the exact published F2 repeat. The official-data fallback is ready if the rules remain unclear near the portal deadline.

Both routes use the six-decimal stationary point of the official starter-only separable quadratic. It predicted the nine later official returns within 2×10⁻¹⁴. An independent campaign sampled this coordinate, which was recovered here from the starter data.

**Week 11 depends on the return.** If the portal returns 10 to the expected precision, use any required subsequent query to check local curvature. Investigate any material mismatch before spending a refinement.

[Week 10 course check and exact batches](../../../submissions/Week_10/README.md) · [Cohort advice and replay](../../../submissions/Week_10/advisory-review.md)

The estimates below use the advisory review's official-data GP comparator and predictive return standard deviation, which includes modelled observation noise. F8 uses the validated starter quadratic and omits a standard deviation because that fit supplies no probabilistic interval. The rows are proposed inputs with model estimates.

In [4]:
from week_10_decision_record import decision_comparison

with pd.option_context("display.max_colwidth", None):
    display(decision_comparison(STAGE, FUNCTION, official))

,Portal input,Starter quadratic (score)
Stage,,
Initial proposal,0.042902-0.140000-0.170186-0.000000-0.946758-0.586941-0.251883-0.480634,9.940899
Advisory experiment,0.100000-0.150000-0.130000-0.150000-0.800000-0.500000-0.200000-0.600000,10.000000
Revised cohort-informed,0.100000-0.150000-0.130000-0.150000-0.800000-0.500000-0.200000-0.600000,10.000000
Official-data fallback,0.100000-0.150000-0.130000-0.150000-0.800000-0.500000-0.200000-0.600000,10.000000


### Conditional peak check

Two Gaussian processes with fitted noise share the same candidate set. A local candidate lies within the stated Euclidean radius of the observed incumbent. The columns report the fraction of joint latent draws in which none of the candidates in that set beats the incumbent by more than the working tolerance. The broader search includes Sobol points, corners, observed coordinates and surrogate-suggested probes. It cannot certify the continuous function or rule out unrepresented peaks. The nine grouped official return holdouts and nominal 90% observation-band counts indicate predictive fit. They do not calibrate the global-maximum event. The tolerance and radius are analysis choices.

In [5]:
radius = 0.12
tolerance = 0.01
peak = peak_review(x, target, later, radius=radius, tolerance=tolerance,
                   length_scales=[0.35, 0.35, 0.35, 0.35, 0.35, 0.35, 0.35, 0.35], noise=[0.0001, 1e-08, 0.12],
                   minimum_distance=model_snapshot["minimum_query_distance"])
print(f"Local radius {radius:g}. Score tolerance {tolerance:g} in {units}.")
print(f"Joint posterior over {peak['candidate_count']} candidates, including {peak['local_count']} local."
      f" {peak['probability_draws']} draws per kernel.")
display(peak["summary"].round(5))
print("Adaptive two-query comparison on the Matérn shortlist")
one_query = peak["acquisitions"]["Matérn 3/2"].nlargest(1, "One-query EI")
print(f"Best one-query input: {one_query.iloc[0]['Input']}")
display(peak["acquisitions"]["Matérn 3/2"].head(5).round(6))

Local radius 0.12. Score tolerance 0.01 in output y.
Joint posterior over 734 candidates, including 159 local. 1024 draws per kernel.


,Local set within tolerance,Broader set within tolerance,Broader gap median,Broader gap 90th pct,Return MAE,Return RMSE,In 90% observation band,Return holdouts,Fitted noise SD,Fit warnings
Model,,,,,,,,,,
Matérn 3/2,0.0,0.0,0.21696,0.42156,0.15495,0.22438,9,9,0.00011,6
RBF,0.0,0.0,0.11804,0.27098,0.26143,0.35384,5,9,0.00011,4


Adaptive two-query comparison on the Matérn shortlist
Best one-query input: 0.090112-0.163656-0.229392-0.006055-0.886365-0.579458-0.214735-0.454178


,Input,Region,Mean,Observed SD,One-query EI,Two-query gain
0,0.090112-0.163656-0.229392-0.006055-0.886365-0...,Local,9.952215,0.097691,0.047179,0.089982
3,0.042902-0.215002-0.170186-0.000000-0.829158-0...,Local,9.968501,0.064677,0.044710,0.087836
1,0.090652-0.161122-0.167903-0.000000-0.898907-0...,Local,9.963261,0.077980,0.046131,0.086833
2,0.143657-0.027306-0.253260-0.047808-0.351922-0...,Elsewhere,9.767045,0.276821,0.045703,0.085939
5,0.041459-0.138471-0.143720-0.036361-0.949989-0...,Local,9.967717,0.060548,0.042714,0.081889


### Strategy for the remaining queries

Prefer the independently fitted response surface and its analytic maximiser after this validation. Keep the two GP estimates as diagnostics. A generic global optimiser would spend opportunities rediscovering a structure that already predicts the returned scores.

The code compares one-query expected improvement in the returned score with a two-query approximation. It conditions on a first outcome, then chooses a different point on a shortlist. Both steps apply the saved minimum-spacing rule. It holds fitted kernel parameters fixed. Refit after each actual result and reassess the next action. The four rounds below are conditional decisions.

| Round | Action given the evidence so far |
| --- | --- |
| 10 | Evaluate the starter-only quadratic's six-decimal stationary point. Both prepared routes use the same input. |
| 11 | If the portal returns 10 to the expected precision, use any required subsequent query to check local curvature. Investigate any material mismatch before spending a refinement. |
| 12 | If an earlier result falsifies the quadratic, locate the residual direction and run a controlled perturbation. Otherwise use a distinct near-maximal point only as needed. |
| 13 | Use the fitted maximiser if still untested, or the best admissible corroborating query warranted by the evidence. |

[Frazier, Bayesian optimisation tutorial](https://arxiv.org/abs/1807.02811) · [Jones, Schonlau and Welch, efficient global optimisation](https://www.schonlau.net/publication/jogo98.pdf)

In [6]:
print(f"{queries_left} query opportunities per function, including Week {WEEK}.")
destination = WEEK_DIR / "weekly-peak-review.json"
summary = {"function": FUNCTION, "week": WEEK, "queries_remaining": queries_left,
           "official_snapshot_sha256": model_snapshot["snapshot_sha256"],
           "candidate_count": peak["candidate_count"], "local_count": peak["local_count"],
           "probability_draws": peak["probability_draws"], "local_radius": radius,
           "tolerance": tolerance, "target_units": units,
           "models": peak["summary"].reset_index().to_dict(orient="records"),
           "matérn_shortlist": peak["acquisitions"]["Matérn 3/2"].to_dict(orient="records")}
if FUNCTION == 8:
    summary["structural_fit"] = quadratic
destination.write_bytes((json.dumps(summary, indent=2, allow_nan=False) + "\n").encode())

4 query opportunities per function, including Week 10.


4471

### Course lifecycle

1. **Define the goal and scope** – choose the next official query while assessing how close the incumbent might be to the peak.
2. **Gather the data** – preserve the official snapshot, the frozen predictions and labelled published comparisons.
3. **Explore the data** – compare the returned result with local and external observations.
4. **Clean and preprocess the data** – verify hashes and portal alignment. Preserve the recorded raw score.
5. **Select and engineer features** – use the original coordinates, with a signed-log target for F1. Structural fits declare their terms separately.
6. **Define the machine-learning task** – regression supports maximisation and assessment of the incumbent.
7. **Partition the data** – withhold each official return and any matching portal coordinate for the predictive check.
8. **Select and train candidate methods** – compare two kernel assumptions and function-specific hypotheses.
9. **Evaluate and interpret the results** – judge forecast reliability and the finite-domain comparison before choosing how to spend the remaining budget.
10. **Deploy and iterate** – deployment is not applicable. The next portal return will update the review and the query plan.